# ODI Complaints Cleaning

This notebook is to explore what cleaning belongs in the shared pipeline and what should stay model specific. We want choices that fit the ODI complaints schema, the documented 2021 changes, and the three model tracks we expect to build.


## Working Rules

- Keep raw meaning before replacing values if possible
- Keep base cleaning conservative, should set a base for other models to work from
- Push text NLP cleanup to the NLP workflow later
- Use explicit flags when a value looks suspicious but not obviously wrong
- Treat `prod_type == 'V'` as the main first-pass modeling cohort since it dominates the product type breakdown


In [ ]:
# Imports
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns
from IPython.display import display

# Preferred pandas and seaborn display parameters
pd.set_option('display.max_columns', 200)
pd.set_option('display.width', 220)
pd.set_option('display.max_colwidth', 120)
plt.style.use('seaborn-v0_8-whitegrid')
sns.set_palette('deep')

## Load The Combined Processed Dataset


In [ ]:
PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / 'data' / 'processed').exists():
    PROJECT_ROOT = PROJECT_ROOT.parent

COMBINED_PATH = PROJECT_ROOT / 'data' / 'processed' / 'odi_complaints_combined.parquet'

raw_df = pd.read_parquet(COMBINED_PATH)
raw_df = raw_df.drop(columns=['source_zip', 'source_file'], errors='ignore')

print("Loaded:", COMBINED_PATH.name)
print("Shape:", raw_df.shape)

## CMPL.txt Notes That Matter

`docs/CMPL.txt` contains the source documentation explaining some of the confusing fields.

In [ ]:
if str(PROJECT_ROOT) not in sys.path:
    sys.path.append(str(PROJECT_ROOT))

from src.data.schema_checks import get_schema_spec

spec = get_schema_spec('complaints')
schema_df = pd.DataFrame(spec['fields'])
desc_map = schema_df.set_index('name')['description'].to_dict()

focus_cols = [
    'yeartxt', 'faildate', 'datea', 'ldate', 'miles', 'injured', 'deaths',
    'veh_speed', 'vin', 'purch_dt', 'manuf_dt', 'state', 'dealer_state',
    'prod_type', 'cdescr'
]

schema_view = schema_df.loc[
    schema_df['name'].isin(focus_cols),
    ['name', 'type', 'size', 'description']
].sort_values('name')

change_notes = pd.DataFrame([
    {'note': 'May-Jun 2021 file update', 'implication': 'Previously blank Y/N fields may now appear as N'},
    {'note': 'May-Jun 2021 file update', 'implication': 'Previously blank numeric fields may now appear as 0'},
    {'note': 'May-Jun 2021 file update', 'implication': 'Manufacturer, make, model, and component values may reflect newer cleanup over time'}
])

print("Schema overview of major features:")
display(schema_view)

print("\nDocumented schema changes:")
display(change_notes)


## Model Framing

The cleaning should support all three model tracks, but not every field should be handled the same way for each one.


In [ ]:
model_plan = pd.DataFrame([
    {
        "model": "structured complaint model",
        "unit_of_analysis": "complaint row",
        "time_anchor": "ldate",
        "initial_cohort": "prod_type='V'",
        "cleaning_focus": "clean structured fields, preserve suspicious rows with flags",
    },
    {
        "model": "time-based early warning model",
        "unit_of_analysis": "make/model/model year/component/time cohort",
        "time_anchor": "ldate",
        "initial_cohort": "prod_type='V'",
        "cleaning_focus": "safe time ordering, leakage-aware date use, stable grouping fields",
    },
    {
        "model": "NLP complaint narrative model",
        "unit_of_analysis": "complaint row",
        "time_anchor": "ldate",
        "initial_cohort": "prod_type='V'",
        "cleaning_focus": "minimal general text cleanup now, NLP-specific text processing later",
    }
])

display(model_plan)

## Vehicle Cohort Check

Before we clean anything aggressively, check whether a vehicle modeling scope is justified.


In [ ]:
prod_counts = raw_df['prod_type'].astype('string').fillna('<NA>').value_counts(dropna=False).rename_axis('prod_type').reset_index(name='row_count')
prod_counts['row_pct'] = (prod_counts['row_count'] / len(raw_df) * 100).round(2)

vehicle_mask = raw_df['prod_type'].astype('string') == 'V'
vehicle_df = raw_df.loc[vehicle_mask].copy()

cohort_summary = pd.DataFrame([
    {
        'view': 'all rows',
        'rows': len(raw_df),
        'row_pct': 100.0,
        'unique_makes': raw_df['maketxt'].nunique(dropna=True),
        'unique_models': raw_df['modeltxt'].nunique(dropna=True),
        'fire_y_pct': round(float((raw_df['fire'].astype('string') == 'Y').mean() * 100), 2),
        'narrative_non_null_pct': round(float(raw_df['cdescr'].notna().mean() * 100), 2)
    },
    {
        'view': "prod_type='V'",
        'rows': len(vehicle_df),
        'row_pct': round(float(len(vehicle_df) / len(raw_df) * 100), 2),
        'unique_makes': vehicle_df['maketxt'].nunique(dropna=True),
        'unique_models': vehicle_df['modeltxt'].nunique(dropna=True),
        'fire_y_pct': round(float((vehicle_df['fire'].astype('string') == 'Y').mean() * 100), 2),
        'narrative_non_null_pct': round(float(vehicle_df['cdescr'].notna().mean() * 100), 2)
    }
])

core_cols = ['maketxt', 'modeltxt', 'yeartxt', 'state', 'miles', 'veh_speed', 'cdescr', 'fire', 'crash']
core_avail = pd.DataFrame({
    'all_rows_non_null_pct': (raw_df[core_cols].notna().mean() * 100).round(2),
    'vehicle_only_non_null_pct': (vehicle_df[core_cols].notna().mean() * 100).round(2)
}).reset_index().rename(columns={'index': 'column'})

print("Dataset product type makeup:")
display(prod_counts)

print("\nSummary of vehicle cohort vs full dataset:")
display(cohort_summary)

print("\nCoverage of major features in vehicle cohort vs full dataset: ")
display(core_avail)

## Field Decision Matrix

This is the working policy table describing the following for each of the major features I've been looking at: 
- what `docs/CMPL.txt` says about it
- the baseline cleaning step I'm taking
- the usage in the structure component model
- the usage in the early warning model
- the usage in the NLP model
- whether the cleaning decision is fully locked in
- what the question is if the decision is still open


In [ ]:
field_decisions = pd.DataFrame([
    {
        'field': 'cmplid',
        'schema_note': desc_map['cmplid'],
        'base_clean': 'keep as string key',
        'structured_model': 'use as row key only',
        'time_model': 'not a feature',
        'nlp_model': 'row key only',
        'current_call': 'settled',
        'open_question': ''
    },
    {
        'field': 'odino',
        'schema_note': desc_map['odino'],
        'base_clean': 'keep as string reference',
        'structured_model': 'do not treat as unique key',
        'time_model': 'case reference only',
        'nlp_model': 'case reference only',
        'current_call': 'settled',
        'open_question': ''
    },
    {
        'field': 'prod_type',
        'schema_note': desc_map['prod_type'],
        'base_clean': 'strip and uppercase',
        'structured_model': 'start with V',
        'time_model': 'start with V',
        'nlp_model': 'start with V',
        'current_call': 'likely settled',
        'open_question': 'Whether any non-V complaints should be folded into later experiments'
    },
    {
        'field': 'maketxt',
        'schema_note': desc_map['maketxt'],
        'base_clean': 'strip and uppercase',
        'structured_model': 'core grouping field',
        'time_model': 'core cohort field',
        'nlp_model': 'metadata only',
        'current_call': 'settled',
        'open_question': ''
    },
    {
        'field': 'modeltxt',
        'schema_note': desc_map['modeltxt'],
        'base_clean': 'strip and uppercase',
        'structured_model': 'core grouping field',
        'time_model': 'core cohort field',
        'nlp_model': 'metadata only',
        'current_call': 'settled',
        'open_question': 'Whether model standardization needs make-aware cleanup later'
    },
    {
        'field': 'yeartxt',
        'schema_note': desc_map['yeartxt'],
        'base_clean': 'convert to Int64 and flag 9999',
        'structured_model': 'usable numeric field',
        'time_model': 'part of cohort definition',
        'nlp_model': 'metadata only',
        'current_call': 'mostly settled',
        'open_question': 'Whether to keep an explicit unknown-year indicator feature'
    },
    {
        'field': 'faildate',
        'schema_note': desc_map['faildate'],
        'base_clean': 'preserve raw datetime',
        'structured_model': 'derived lags only if sane',
        'time_model': 'secondary time field',
        'nlp_model': 'metadata only',
        'current_call': 'open',
        'open_question': 'How to use incident date when it conflicts slightly with admin dates'
    },
    {
        'field': 'datea',
        'schema_note': desc_map['datea'],
        'base_clean': 'preserve raw datetime',
        'structured_model': 'probably admin-only',
        'time_model': 'not main anchor',
        'nlp_model': 'metadata only',
        'current_call': 'open',
        'open_question': 'Whether this field is worth keeping in final model tables'
    },
    {
        'field': 'ldate',
        'schema_note': desc_map['ldate'],
        'base_clean': 'preserve raw datetime',
        'structured_model': 'main time anchor',
        'time_model': 'main time anchor',
        'nlp_model': 'main time anchor',
        'current_call': 'settled',
        'open_question': ''
    },
    {
        'field': 'miles',
        'schema_note': desc_map['miles'],
        'base_clean': 'convert to Int64 and flag highs',
        'structured_model': 'likely keep with flags or transforms',
        'time_model': 'cohort feature candidate',
        'nlp_model': 'metadata only',
        'current_call': 'open',
        'open_question': 'Whether very high mileage should stay, cap, or become missing'
    },
    {
        'field': 'veh_speed',
        'schema_note': desc_map['veh_speed'],
        'base_clean': 'convert to Int64 and flag 999/high values',
        'structured_model': 'keep if interpretable',
        'time_model': 'probably weak feature',
        'nlp_model': 'metadata only',
        'current_call': 'open',
        'open_question': 'Whether >200 should stay flagged only or become missing'
    },
    {
        'field': 'injured',
        'schema_note': desc_map['injured'],
        'base_clean': 'convert to Int64 and flag 99',
        'structured_model': 'possible target-adjacent field',
        'time_model': 'severity aggregate candidate',
        'nlp_model': 'possible label depending on task',
        'current_call': 'open',
        'open_question': 'How much to trust zeros after the 2021 file change'
    },
    {
        'field': 'deaths',
        'schema_note': desc_map['deaths'],
        'base_clean': 'convert to Int64 and flag 99',
        'structured_model': 'possible target-adjacent field',
        'time_model': 'severity aggregate candidate',
        'nlp_model': 'possible label depending on task',
        'current_call': 'open',
        'open_question': 'How much to trust zeros after the 2021 file change'
    },
    {
        'field': 'state',
        'schema_note': desc_map['state'],
        'base_clean': 'strip, uppercase, validate',
        'structured_model': 'coarse geography candidate',
        'time_model': 'regional aggregate candidate',
        'nlp_model': 'metadata only',
        'current_call': 'mostly settled',
        'open_question': 'Whether to collapse to region later'
    },
    {
        'field': 'dealer_state',
        'schema_note': desc_map['dealer_state'],
        'base_clean': 'strip, uppercase, validate',
        'structured_model': 'optional metadata',
        'time_model': 'likely low priority',
        'nlp_model': 'metadata only',
        'current_call': 'open',
        'open_question': 'Whether dealer fields are useful enough to keep in first-pass models'
    },
    {
        'field': 'vin',
        'schema_note': desc_map['vin'],
        'base_clean': 'strip and uppercase',
        'structured_model': 'feature quality flag only',
        'time_model': 'not a feature',
        'nlp_model': 'not a feature',
        'current_call': 'mostly settled',
        'open_question': 'Whether VIN presence or length quality is predictive'
    },
    {
        'field': 'cdescr',
        'schema_note': desc_map['cdescr'],
        'base_clean': 'strip only',
        'structured_model': 'not in first structured baseline',
        'time_model': 'maybe aggregate text signals later',
        'nlp_model': 'main input field',
        'current_call': 'settled for now',
        'open_question': 'NLP-specific cleaning belongs later'
    }
])

display(field_decisions)

## Set Up Layered Data Frames

Keep the stages separate so the review cells can show exactly what changed and when.


In [ ]:
std_df = raw_df.copy()
flag_df = pd.DataFrame(index=raw_df.index)
candidate_df = None
YN_COLS = [
    'crash',
    'fire',
    'police_rpt_yn',
    'orig_owner_yn',
    'anti_brakes_yn',
    'cruise_cont_yn',
    'orig_equip_yn',
    'repaired_yn',
    'medical_attn',
    'vehicles_towed_yn',
]
UPPER_COLS = [
    'mfr_name',
    'maketxt',
    'modeltxt',
    'compdesc',
    'city',
    'state',
    'vin',
    'cmpl_type',
    'dealer_name',
    'dealer_city',
    'dealer_state',
    'prod_type',
] + YN_COLS
STRIP_ONLY_COLS = [
    'cdescr',
    'fuel_sys',
    'fuel_type',
    'trans_type',
    'drive_train',
    'dot',
    'tire_size',
    'loc_of_tire',
    'tire_fail_type',
    'dealer_zip',
    'dealer_tel',
]
INT_COLS = [
    'yeartxt',
    'injured',
    'deaths',
    'miles',
    'occurences',
    'num_cyls',
    'veh_speed',
]
DATE_STR_COLS = ['purch_dt', 'manuf_dt']
VALID_PROD_TYPES = {'V', 'T', 'E', 'C'}
POSTAL_CODES = {
    'AL', 'AK', 'AZ', 'AR', 'CA', 'CO', 'CT', 'DE', 'FL', 'GA', 'HI', 'ID',
    'IL', 'IN', 'IA', 'KS', 'KY', 'LA', 'ME', 'MD', 'MA', 'MI', 'MN', 'MS',
    'MO', 'MT', 'NE', 'NV', 'NH', 'NJ', 'NM', 'NY', 'NC', 'ND', 'OH', 'OK',
    'OR', 'PA', 'RI', 'SC', 'SD', 'TN', 'TX', 'UT', 'VT', 'VA', 'WA', 'WV',
    'WI', 'WY', 'DC', 'PR', 'VI', 'GU', 'AS', 'MP', 'FM', 'MH', 'PW', 'AE',
    'AA', 'AP'
}

# -----------------------------------------------------------------------------
# Small cleaning helpers
# -----------------------------------------------------------------------------
def strip_text(series):
    text = series.astype('string').str.strip()
    return text.replace({'': pd.NA})


def upper_text(series):
    return strip_text(series).str.upper()


def to_int(series):
    return pd.to_numeric(strip_text(series), errors='coerce').astype('Int64')


def to_dt(series):
    return pd.to_datetime(strip_text(series), format='%Y%m%d', errors='coerce')


# One helper here beats four copy-paste compare blocks
def compare_view(mask, cols, n=10):
    raw_view = raw_df.loc[mask, cols].add_prefix('raw_')
    std_view = std_df.loc[mask, cols].add_prefix('std_')
    cand_view = candidate_df.loc[mask, cols].add_prefix('cand_')
    return raw_view.join(std_view).join(cand_view).head(n)

## Base Standardization

This stage should be safe across all three model tracks. It standardizes text and parses obvious numeric or date fields, but it does not yet erase values just because they look odd.


In [ ]:
text_rows = []
dtype_rows = []

# Format to uppercase and change dtype of all UPPER_COL features
for col in UPPER_COLS:
    if col not in std_df.columns:
        continue
    before = std_df[col].copy()
    std_df[col] = upper_text(std_df[col])
    changed = int(
        before.astype('string')
        .fillna('<NA>')
        .ne(std_df[col].astype('string').fillna('<NA>'))
        .sum()
    )
    text_rows.append(
        {
            'column': col,
            'action': 'strip + upper',
            'changed_rows': changed,
            'null_after': int(std_df[col].isna().sum()),
        }
    )

# Strip whitespace and format all listed features
for col in STRIP_ONLY_COLS:
    if col not in std_df.columns:
        continue
    before = std_df[col].copy()
    std_df[col] = strip_text(std_df[col])
    changed = int(
        before.astype('string')
        .fillna('<NA>')
        .ne(std_df[col].astype('string').fillna('<NA>'))
        .sum()
    )
    text_rows.append(
        {
            'column': col,
            'action': 'strip only',
            'changed_rows': changed,
            'null_after': int(std_df[col].isna().sum()),
        }
    )

# Change all columns that should be int
for col in INT_COLS:
    if col not in std_df.columns:
        continue
    before_dtype = str(std_df[col].dtype)
    before_non_null = int(std_df[col].notna().sum())
    std_df[col] = to_int(std_df[col])
    dtype_rows.append(
        {
            'column': col,
            'action': 'to Int64',
            'dtype_before': before_dtype,
            'dtype_after': str(std_df[col].dtype),
            'non_null_before': before_non_null,
            'non_null_after': int(std_df[col].notna().sum()),
        }
    )

# Format data columns as proper datetimes
for col in DATE_STR_COLS:
    if col not in std_df.columns:
        continue
    before_dtype = str(std_df[col].dtype)
    before_non_null = int(std_df[col].notna().sum())
    std_df[col] = to_dt(std_df[col])
    dtype_rows.append(
        {
            'column': col,
            'action': 'to datetime',
            'dtype_before': before_dtype,
            'dtype_after': str(std_df[col].dtype),
            'non_null_before': before_non_null,
            'non_null_after': int(std_df[col].notna().sum()),
        }
    )

text_log = pd.DataFrame(text_rows).sort_values(
    ['changed_rows', 'column'], ascending=[False, True]
)
dtype_log = pd.DataFrame(dtype_rows)

print("Changes from text actions: ")
display(text_log[text_log['changed_rows'] > 0])

print("\nChanges from data type changes: ")
display(dtype_log)

## Issue Flags

Flags both sentinel values noted in `docs/CMPL.txt` and invalid range/values for features. We want to separate values that just seem suspicious and those that definitely need to be replaced or dropped.

In [ ]:
current_year = pd.Timestamp.today().year

flag_df['flag_prod_type_bad'] = std_df['prod_type'].notna() & ~std_df['prod_type'].isin(VALID_PROD_TYPES)
flag_df['flag_year_unknown'] = std_df['yeartxt'].eq(9999)
flag_df['flag_year_out_of_range'] = std_df['yeartxt'].notna() & ~std_df['yeartxt'].between(1900, current_year + 1)
flag_df['flag_speed_999'] = std_df['veh_speed'].eq(999)
flag_df['flag_speed_high'] = std_df['veh_speed'].gt(200) & ~std_df['veh_speed'].eq(999)
flag_df['flag_miles_high'] = std_df['miles'].gt(500000)
flag_df['flag_injured_99'] = std_df['injured'].eq(99)
flag_df['flag_deaths_99'] = std_df['deaths'].eq(99)
flag_df['flag_state_bad'] = std_df['state'].notna() & ~std_df['state'].isin(POSTAL_CODES)
flag_df['flag_dealer_state_bad'] = std_df['dealer_state'].notna() & ~std_df['dealer_state'].isin(POSTAL_CODES)
flag_df['flag_vin_len_bad'] = std_df['vin'].notna() & std_df['vin'].str.len().ne(11)
flag_df['flag_fail_after_added'] = std_df['faildate'].notna() & std_df['datea'].notna() & (std_df['faildate'] > std_df['datea'])
flag_df['flag_fail_after_received'] = std_df['faildate'].notna() & std_df['ldate'].notna() & (std_df['faildate'] > std_df['ldate'])
flag_df['flag_added_before_received'] = std_df['datea'].notna() & std_df['ldate'].notna() & (std_df['datea'] < std_df['ldate'])
flag_df['flag_date_order_bad'] = flag_df[['flag_fail_after_added', 'flag_fail_after_received', 'flag_added_before_received']].any(axis=1)

flag_summary = flag_df.sum().rename('row_count').reset_index().rename(columns={'index': 'flag'})
flag_summary['row_pct'] = (flag_summary['row_count'] / len(std_df) * 100).round(4)
flag_summary = flag_summary.sort_values(['row_count', 'flag'], ascending=[False, True])

display(flag_summary)

## 2021 Changelog Check

`CMPL.txt` warns that some blank numeric fields became 0 and some blank Y/N fields became N after the 2021 file change. This does not prove the current values are wrong, but it does mean we should be cautious when treating zeros and N as equally reliable across time.

In [ ]:
zero_watch = (
    std_df.assign(received_year=std_df['ldate'].dt.year)
    .groupby('received_year')[['injured', 'deaths', 'miles', 'veh_speed']]
    .agg(lambda s: round(float((s == 0).mean() * 100), 2))
    .reset_index()
)

yn_watch = (
    std_df.assign(received_year=std_df['ldate'].dt.year)
    .groupby('received_year')[['crash', 'fire']]
    .agg(lambda s: round(float((s == 'N').mean() * 100), 2))
    .reset_index()
)

print("Percent of rows equal to zero by received year")
display(zero_watch)

print("Percent of rows equal to N by received year")
display(yn_watch)


## High Confidence Candidate Decisions

These are decisions that are locked in and shouldn't have any uncertainty in them. Potential suspicious values are not included here.

In [ ]:
candidate_df = std_df.copy()

candidate_df.loc[flag_df['flag_prod_type_bad'], 'prod_type'] = pd.NA
candidate_df.loc[flag_df['flag_year_unknown'] | flag_df['flag_year_out_of_range'], 'yeartxt'] = pd.NA
candidate_df.loc[flag_df['flag_speed_999'], 'veh_speed'] = pd.NA
candidate_df.loc[flag_df['flag_injured_99'], 'injured'] = pd.NA
candidate_df.loc[flag_df['flag_deaths_99'], 'deaths'] = pd.NA
candidate_df.loc[flag_df['flag_state_bad'], 'state'] = pd.NA
candidate_df.loc[flag_df['flag_dealer_state_bad'], 'dealer_state'] = pd.NA

candidate_rules = pd.DataFrame([
    {
        'field': 'prod_type',
        'rule': 'invalid product type to missing',
        'applied_rows': int(flag_df['flag_prod_type_bad'].sum()),
        'why': 'schema-coded field with a small and explicit valid set'
    },
    {
        'field': 'yeartxt',
        'rule': '9999 and out-of-range values to missing',
        'applied_rows': int((flag_df['flag_year_unknown'] | flag_df['flag_year_out_of_range']).sum()),
        'why': 'CMPL.txt documents 9999 as unknown and extreme years are not usable model years'
    },
    {
        'field': 'veh_speed',
        'rule': '999 sentinel to missing',
        'applied_rows': int(flag_df['flag_speed_999'].sum()),
        'why': '999 behaves like a placeholder value'
    },
    {
        'field': 'injured',
        'rule': '99 sentinel to missing',
        'applied_rows': int(flag_df['flag_injured_99'].sum()),
        'why': '99 is not a plausible person count here'
    },
    {
        'field': 'deaths',
        'rule': '99 sentinel to missing',
        'applied_rows': int(flag_df['flag_deaths_99'].sum()),
        'why': '99 is not a plausible fatality count here'
    },
    {
        'field': 'state / dealer_state',
        'rule': 'invalid codes to missing',
        'applied_rows': int(flag_df['flag_state_bad'].sum() + flag_df['flag_dealer_state_bad'].sum()),
        'why': 'invalid location codes are not analytically useful as geography'
    },
    {
        'field': 'miles',
        'rule': 'flag only for now',
        'applied_rows': 0,
        'why': 'very high mileage can still be real and needs model-specific handling'
    },
    {
        'field': 'date fields',
        'rule': 'flag only for now',
        'applied_rows': 0,
        'why': 'slight chronology mismatches may be administrative rather than wrong'
    },
    {
        'field': 'vin',
        'rule': 'flag only for now',
        'applied_rows': 0,
        'why': 'CMPL.txt defines VIN as CHAR(11) in this extract, but odd lengths still deserve review'
    }
])

display(candidate_rules)

## Impact Summary

This compares the raw data, the standardized layer, and the candidate cleaned layer. If null rates barely move, that is still useful because it tells us the current cleaning is mostly about type correctness and flagging rather than value replacement.

In [ ]:
compare_cols = [
    'yeartxt',
    'miles',
    'veh_speed',
    'injured',
    'deaths',
    'purch_dt',
    'manuf_dt',
    'state',
    'dealer_state',
    'maketxt',
    'modeltxt',
    'city',
    'dealer_city'
]

null_compare = pd.DataFrame({
    'raw_null_pct': (raw_df[compare_cols].isna().mean() * 100).round(2),
    'std_null_pct': (std_df[compare_cols].isna().mean() * 100).round(2),
    'candidate_null_pct': (candidate_df[compare_cols].isna().mean() * 100).round(2)
}).reset_index().rename(columns={'index': 'column'})

dtype_compare = pd.DataFrame({
    'raw_dtype': raw_df[compare_cols].dtypes.astype('string'),
    'std_dtype': std_df[compare_cols].dtypes.astype('string'),
    'candidate_dtype': candidate_df[compare_cols].dtypes.astype('string')
}).reset_index().rename(columns={'index': 'column'})

checks = pd.DataFrame([
        {'metric': 'rows after candidate cleaning', 'value': len(candidate_df)},
        {'metric': 'columns in candidate frame', 'value': candidate_df.shape[1]},
        {'metric': 'cmplid unique after candidate cleaning', 'value': candidate_df['cmplid'].nunique(dropna=True)},
        {'metric': 'odino unique after candidate cleaning', 'value': candidate_df['odino'].nunique(dropna=True)},
        {'metric': 'full-row duplicates after candidate cleaning', 'value': int(candidate_df.duplicated().sum())}
])

print("Null percentages by layer: ")
display(null_compare.sort_values('candidate_null_pct', ascending=False))

print("\nData type by layer: ")
display(dtype_compare)

print("\nGeneral checks after cleaning: ")
display(checks)

plot_df = null_compare.melt(id_vars='column', var_name='stage', value_name='null_pct')

plt.figure(figsize=(12, 8))
sns.barplot(data=plot_df, x='null_pct', y='column', hue='stage')
plt.title("Null Percentage by Cleaning Stage")
plt.xlabel("Null Percentage")
plt.ylabel("")
plt.tight_layout()
plt.show()

## Review Raw Vs Candidate Values

These are the cells to study before we promote any new rule from "flag only" to "replace in the shared pipeline".


In [ ]:
review_cols = [
    'cmplid',
    'odino',
    'prod_type',
    'maketxt',
    'modeltxt',
    'yeartxt',
    'state',
    'dealer_state',
    'vin',
    'faildate',
    'datea',
    'ldate',
    'miles',
    'veh_speed',
    'injured',
    'deaths',
    'cdescr'
]

print("Chronology rows to review:")
display(compare_view(flag_df['flag_date_order_bad'], review_cols, n=10))

print("\nBad state rows to review:")
display(compare_view(flag_df['flag_state_bad'] | flag_df['flag_dealer_state_bad'], review_cols, n=10))

print("\nModel year rows to review:")
display(compare_view(flag_df['flag_year_unknown'] | flag_df['flag_year_out_of_range'], review_cols, n=10))

print("\nSpeed and miles rows to review:")
display(compare_view(flag_df['flag_speed_999'] | flag_df['flag_speed_high'] | flag_df['flag_miles_high'], review_cols, n=10))

print("\nVIN rows to review:")
display(compare_view(flag_df['flag_vin_len_bad'], review_cols, n=10))

## Vehicle-Ready Preview

Using only the vehicle product type is where our first model pass will start, then we can expand from there if we want to capture more variety. The point here is to see what the vehicle only table looks like after moderate cleaning.

In [ ]:
vehicle_candidate = candidate_df.loc[candidate_df['prod_type'] == 'V'].copy()

vehicle_ready = pd.DataFrame([
    {'metric': 'vehicle rows', 'value': len(vehicle_candidate)},
    {'metric': 'vehicle row share pct', 'value': round(float(len(vehicle_candidate) / len(candidate_df) * 100), 2)},
    {'metric': 'vehicle unique makes', 'value': vehicle_candidate['maketxt'].nunique(dropna=True)},
    {'metric': 'vehicle unique models', 'value': vehicle_candidate['modeltxt'].nunique(dropna=True)}
])

vehicle_core = pd.DataFrame({
    'vehicle_non_null_pct': (vehicle_candidate[[
        'maketxt',
        'modeltxt',
        'yeartxt',
        'state',
        'miles',
        'veh_speed',
        'injured',
        'deaths',
        'cdescr'
    ]].notna().mean() * 100).round(2)
}).reset_index().rename(columns={'index': 'column'})

vehicle_flags = pd.DataFrame({
    'flag': flag_df.columns,
    'vehicle_row_pct': [round(float(flag_df.loc[vehicle_candidate.index, col].mean() * 100), 4) for col in flag_df.columns]
}).sort_values('vehicle_row_pct', ascending=False)

print("General vehicle cohort overview:")
display(vehicle_ready)

print("\nCore vehicle features coverage:")
display(vehicle_core)

print("\nPercent of vehicle cohort rows flagged bad or suspicious")
display(vehicle_flags)

## Decision Queue

These are the next issues to settle before the final `src/` cleaning module is written.


In [ ]:
decision_queue = pd.DataFrame([
    {
        'topic': 'date chronology',
        'current_position': 'keep raw dates and keep flags',
        'why_it_is_open': 'small mismatches may be administrative rather than wrong',
        'next_check': 'derive lag features from ldate and faildate and inspect negative lags'
    },
    {
        'topic': 'very high mileage',
        'current_position': 'flag only',
        'why_it_is_open': 'high mileage can be real and may still be predictive',
        'next_check': 'inspect high-mileage rows by make, model, and complaint type'
    },
    {
        'topic': 'vehicle speed above 200',
        'current_position': 'flag only',
        'why_it_is_open': 'possible data entry issue but not as obviously sentinel-like as 999',
        'next_check': 'inspect the raw complaints and decide whether to keep, cap, or null'
    },
    {
        'topic': 'zeros after the 2021 file change',
        'current_position': 'treat as observed but suspicious in some fields',
        'why_it_is_open': 'CMPL.txt warns that blanks may now appear as 0 or N',
        'next_check': 'decide field by field whether zero is safe, ambiguous, or target-adjacent'
    },
    {
        'topic': 'dealer fields',
        'current_position': 'keep in base clean',
        'why_it_is_open': 'they may not matter enough for first-pass models',
        'next_check': 'decide whether to exclude them from the first structured baseline'
    },
    {
        'topic': 'vehicle-only focus',
        'current_position': 'strong first-pass choice',
        'why_it_is_open': 'non-vehicle product types are structurally different',
        'next_check': 'confirm whether any later model should bring T, E, or C back in'
    }
])

display(decision_queue)

## Scratchpad


In [ ]:
# Example checks
# compare_view(flag_df['flag_speed_high'], review_cols, n=25)
# candidate_df.loc[candidate_df['prod_type'] == 'V', ['maketxt', 'modeltxt', 'compdesc']].head(20)
# candidate_df.groupby('prod_type').size().sort_values(ascending=False)
# candidate_df.loc[flag_df['flag_miles_high'], ['maketxt', 'modeltxt', 'miles', 'cdescr']].head(20)
